# 🎯 Task 2: Customer Segmentation & Behavioral Analytics

## 📌 Objective
Segment customers based on **demographics** (Age, Gender, Annual Income) and **purchasing behaviors** (Recency, Frequency, Monetary value, Spending Score) using **Unsupervised Machine Learning (K-Means Clustering)**.

### 📊 Business Questions Addressed:
1. Who are our most valuable, high-spending customers (*Champions*)?
2. Which customer segment is at risk of churning due to low recency?
3. How do purchasing preferences vary across age groups and income brackets?
4. What targeted marketing strategies maximize Customer Lifetime Value (CLV)?

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Libraries loaded successfully.")

## 1. Data Ingestion & Inspection

In [2]:
df = pd.read_csv('Customer_Segmentation_Data.csv')
print(f"Total Customers: {df.shape[0]} | Columns: {df.shape[1]}")
df.head()

## 2. Feature Selection & Standardization
K-Means relies on Euclidean distance. Features with larger ranges (e.g., Monetary Spend up to $25,000) would dominate smaller features (e.g., Frequency from 1-35). We apply `StandardScaler` to bring all features to a mean of 0 and standard deviation of 1.

In [3]:
feature_cols = [
    'Age',
    'AnnualIncome_k',
    'SpendingScore',
    'RecencyDays',
    'FrequencyOrders',
    'MonetaryTotalSpend'
]

X = df[feature_cols]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## 3. Determining the Optimal Number of Clusters (Elbow Method & Silhouette Analysis)

In [4]:
wcss = []
sil_scores = []
k_values = range(2, 9)

for k in k_values:
    km = KMeans(n_clusters=k, init='k-means++', max_iter=300, n_init=10, random_state=42)
    km.fit(X_scaled)
    wcss.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(k_values, wcss, marker='o', color='#3b82f6', lw=2)
ax[0].set_title('Elbow Method (WCSS vs. K)', fontsize=13, fontweight='bold')
ax[0].set_xlabel('Number of Clusters (K)')
ax[0].set_ylabel('Inertia / WCSS')

ax[1].plot(k_values, sil_scores, marker='s', color='#10b981', lw=2)
ax[1].set_title('Silhouette Score vs. K', fontsize=13, fontweight='bold')
ax[1].set_xlabel('Number of Clusters (K)')
ax[1].set_ylabel('Silhouette Score')
plt.tight_layout()
plt.show()

# Optimal K = 4 shows the clear inflection 'elbow' point with high cohesion.

## 4. Training Final Model (K=4) & Persona Profiling

In [5]:
kmeans_model = KMeans(n_clusters=4, init='k-means++', max_iter=300, n_init=10, random_state=42)
df['Cluster'] = kmeans_model.fit_predict(X_scaled)

# Profiling cluster traits
profile_summary = df.groupby('ClusterName')[feature_cols].mean().round(2)
profile_summary['Count'] = df.groupby('ClusterName').size()
profile_summary['Share_%'] = (profile_summary['Count'] / len(df) * 100).round(1)
profile_summary

## 5. Strategic Recommendations
- **Champions (High Value):** VIP loyalty rewards, exclusive early access to product launches, zero discounts needed.
- **Loyal Bargain Seekers:** Flash discounts, bundle promotions (buy 2 get 1 free), clearance sales.
- **Young Tech Explorers:** Social media ads, mobile push notifications, cutting-edge gadget marketing.
- **At-Risk / Low Engagement:** Re-engagement drip campaigns, 'We miss you' discount vouchers, feedback surveys.